#Analise e Comparação Estatística entre Métodos de Interpretabildaide / Métricas Cálculadas

In [ ]:
# == Carrega as métricas, limpa os dados e ajusta colunas dos DFs por método/classificador ==

import os
import pandas as pd

DATA_BASE  = "WDBC"
CLASSIFIER = ["RANDOM", "XGBOOST"]
METHOD     = ["LIME", "SHAP", "ANCHOR", "PFI", "SURROGATE"]

# Mapeia cada METHOD para (prefixo_do_df, nome_do_csv)
METHOD_CONFIG = {
    "LIME":                  ("df_lime_metrics",           "LIME_METRICS.csv"),
    "SHAP":                  ("df_shape_metrics",          "SHAP_METRICS.csv"),
    "ANCHOR":                ("df_anchor_metrics",         "ANCHOR_METRICS.csv"),
    "PFI":                   ("df_permutation_metrics",    "PFI_METRICS.csv"),
    "SURROGATE":             ("df_surrogate_tree_metrics", "SURROGATE_METRICS.csv"),
}


def verificar_ou_carregar(nome_df: str, nome_arquivo: str):
    """
    Verifica se o DataFrame já existe no ambiente global.
    Caso não exista, carrega o arquivo CSV correspondente.
    """
    if os.path.exists(nome_arquivo):
        print(f"📂 Carregando {nome_arquivo} para {nome_df}...")
        globals()[nome_df] = pd.read_csv(nome_arquivo)
    else:
        print(f"⚠ Arquivo {nome_arquivo} não encontrado. {nome_df} não será criado.")


# =========================
# 1) Carregamento
# =========================

for classifier in CLASSIFIER:
    print(f"Processando classifier: {classifier}")
    for method in METHOD:
        print(f"  Processando method: {method}")
        base_path = f"/5_xAI/databases/{DATA_BASE}/notebooks/Methods/{DATA_BASE}_{method}_{classifier}/"
        print(f"    Base path: {base_path}")

        df_prefix, csv_name = METHOD_CONFIG[method]

        nome_df     = f"{df_prefix}_{classifier}"          # ex: df_lime_metrics_RANDOM
        nome_arquivo = os.path.join(base_path, csv_name)   # ex: .../WDBC_LIME_RANDOM/LIME_METRICS.csv

        verificar_ou_carregar(nome_df, nome_arquivo)


# =========================
# 2) Normalização de colunas
# =========================
# Regras:
# - dfs LIME:        renomear 'Unnamed: 0' -> 'item' e 'instancia' -> 'id'
# - dfs SHAP:        renomear 'instancia' -> 'id' e adicionar 'item' no começo (0..n-1)
# - dfs ANCHOR:      renomear 'instancia' -> 'id' e adicionar 'item' no começo (0..n-1)
# - dfs PERMUTATION: renomear 'Unnamed: 0' -> 'item' e 'instancia' -> 'id'
# - dfs SURROGATE:   renomear 'instancia' -> 'id' e adicionar 'item' no começo (0..n-1)

def _reordenar_item_primeiro(df: pd.DataFrame) -> pd.DataFrame:
    """Garante que 'item' seja a primeira coluna, se existir."""
    if "item" in df.columns:
        cols = ["item"] + [c for c in df.columns if c != "item"]
        df = df[cols]
    return df


for classifier in CLASSIFIER:
    # ---------- LIME ----------
    lime_name = f"df_lime_metrics_{classifier}"
    if lime_name in globals():
        df = globals()[lime_name].copy()

        # renomear Unnamed: 0 -> item
        for col in df.columns:
            if col.strip().lower().startswith("unnamed"):
                df = df.rename(columns={col: "item"})
                break

        # renomear instancia -> id
        if "instancia" in df.columns:
            df = df.rename(columns={"instancia": "id"})

        df = _reordenar_item_primeiro(df)
        globals()[lime_name] = df

    # ---------- SHAP ----------
    shap_name = f"df_shape_metrics_{classifier}"
    if shap_name in globals():
        df = globals()[shap_name].copy()

        # renomear instancia -> id
        if "instancia" in df.columns:
            df = df.rename(columns={"instancia": "id"})

        # adicionar item no começo (0..n-1) se não existir
        if "item" not in df.columns:
            df.insert(0, "item", range(len(df)))

        df = _reordenar_item_primeiro(df)
        globals()[shap_name] = df

    # ---------- ANCHOR ----------
    anchor_name = f"df_anchor_metrics_{classifier}"
    if anchor_name in globals():
        df = globals()[anchor_name].copy()

        # renomear instancia -> id
        if "instancia" in df.columns:
            df = df.rename(columns={"instancia": "id"})

        # adicionar item no começo (0..n-1) se não existir
        if "item" not in df.columns:
            df.insert(0, "item", range(len(df)))

        df = _reordenar_item_primeiro(df)
        globals()[anchor_name] = df

    # ---------- PERMUTATION ----------
    perm_name = f"df_permutation_metrics_{classifier}"
    if perm_name in globals():
        df = globals()[perm_name].copy()

        # renomear Unnamed: 0 -> item
        for col in df.columns:
            if col.strip().lower().startswith("unnamed"):
                df = df.rename(columns={col: "item"})
                break

        # renomear instancia -> id
        if "instancia" in df.columns:
            df = df.rename(columns={"instancia": "id"})

        df = _reordenar_item_primeiro(df)
        globals()[perm_name] = df

    # ---------- SURROGATE ----------
    sur_name = f"df_surrogate_tree_metrics_{classifier}"
    if sur_name in globals():
        df = globals()[sur_name].copy()

        # renomear instancia -> id
        if "instancia" in df.columns:
            df = df.rename(columns={"instancia": "id"})

        # adicionar item no começo (0..n-1) se não existir
        if "item" not in df.columns:
            df.insert(0, "item", range(len(df)))

        df = _reordenar_item_primeiro(df)
        globals()[sur_name] = df

print("✅ Ajustes de colunas concluídos.")


In [ ]:
feat_names = [
    "fidelity(%)","infidelity(%)","faithfulness(%)","simplicity(%)",
    "consistency(%)","robustez(%)","completeness(%)","selectivity(%)",
    "soundness(%)","directional soundness(%)","stability(%)",
    "sufficiency-1(%)","sufficiency-5(%)","sufficiency-10(%)","sufficiency-20(%)"
]

def preparar_metrics_calc(df):
    df_calc = df[feat_names]
    df_calc = df_calc.iloc[:-2].copy()
    return df_calc

# ----------------- LIME -----------------
df_lime_metrics_RANDOM_calc   = preparar_metrics_calc(df_lime_metrics_RANDOM)
df_lime_metrics_XGBOOST_calc  = preparar_metrics_calc(df_lime_metrics_XGBOOST)

# ----------------- SHAP -----------------
df_shap_metrics_RANDOM_calc   = preparar_metrics_calc(df_shape_metrics_RANDOM)
df_shap_metrics_XGBOOST_calc  = preparar_metrics_calc(df_shape_metrics_XGBOOST)

# ----------------- ANCHOR -----------------
df_anchor_metrics_RANDOM_calc  = preparar_metrics_calc(df_anchor_metrics_RANDOM)
df_anchor_metrics_XGBOOST_calc = preparar_metrics_calc(df_anchor_metrics_XGBOOST)

# ----------------- SURROGATE TREE -----------------
df_surrogate_tree_metrics_RANDOM_calc  = preparar_metrics_calc(df_surrogate_tree_metrics_RANDOM)
df_surrogate_tree_metrics_XGBOOST_calc = preparar_metrics_calc(df_surrogate_tree_metrics_XGBOOST)

# ----------------- PERMUTATION (PFI) -----------------
df_permutation_metrics_RANDOM_calc  = preparar_metrics_calc(df_permutation_metrics_RANDOM)
df_permutation_metrics_XGBOOST_calc = preparar_metrics_calc(df_permutation_metrics_XGBOOST)


In [ ]:
from scipy.stats import wilcoxon
import numpy as np
import pandas as pd

feat_names = [
    "fidelity(%)","infidelity(%)","faithfulness(%)","simplicity(%)",
    "consistency(%)","robustez(%)","completeness(%)","selectivity(%)",
    "soundness(%)","directional soundness(%)","stability(%)",
    "sufficiency-1(%)","sufficiency-5(%)","sufficiency-10(%)","sufficiency-20(%)"
]

alpha = 0.05  # nível de significância

def calcular_wilcoxon_metrics(df_random_calc, df_xgboost_calc, metodo_nome):
    """
    Executa o Wilcoxon pareado RANDOM vs XGBOOST para todas as feat_names
    e devolve um DataFrame de resultados no mesmo formato que você já usa.
    """
    resultados = []

    for col in feat_names:
        # converte para float e garante arrays
        serie_r = df_random_calc[col].astype(float).to_numpy()
        serie_x = df_xgboost_calc[col].astype(float).to_numpy()

        # máscara para garantir pares válidos (sem NaN)
        mask_valid = ~np.isnan(serie_r) & ~np.isnan(serie_x)
        serie_r_v = serie_r[mask_valid]
        serie_x_v = serie_x[mask_valid]

        # se não tiver pares válidos, pula
        if len(serie_r_v) == 0:
            continue

        # Wilcoxon pareado
        stat, p = wilcoxon(serie_r_v, serie_x_v, zero_method="wilcox")

        # inicializa campos das novas colunas como vazio
        media_geral   = ""
        mediana_geral = ""
        media_r       = ""
        mediana_r     = ""
        media_x       = ""
        mediana_x     = ""

        # trata NaN explicitamente
        if np.isnan(p):
            p_str = "0"
            p_num = 0.0
            conclusao = "Não Diferentes (p ≥ 0.05)"
        else:
            p_str = f"{p:.4f}"
            p_num = float(f"{p:.4f}")
            if p < alpha:
                conclusao = "Diferentes (p < 0.05)"
            else:
                conclusao = "Não Diferentes (p ≥ 0.05)"

        # === cálculo das médias/medianas com 3 casas ===
        if conclusao.startswith("Não Diferentes"):
            # usa os valores combinados dos dois classificadores
            ambos = np.concatenate([serie_r_v, serie_x_v])
            media_geral   = f"{np.mean(ambos):.3f}"
            mediana_geral = f"{np.median(ambos):.3f}"
        else:
            # usa médias/medianas separadas para cada classificador
            media_r   = f"{np.mean(serie_r_v):.3f}"
            mediana_r = f"{np.median(serie_r_v):.3f}"
            media_x   = f"{np.mean(serie_x_v):.3f}"
            mediana_x = f"{np.median(serie_x_v):.3f}"

        resultados.append({
            "metodo": metodo_nome,
            "métrica": col,
            "estatística": stat,
            "p-value_num": f"{p_num:.4f}",   # valor numérico (já tratado)
            "conclusão": conclusao,
            # novas colunas
            "media_geral_3casas":    media_geral,     # usada quando NÃO diferentes
            "mediana_geral_3casas":  mediana_geral,   # usada quando NÃO diferentes
            "media_RANDOM_3casas":   media_r,         # usadas quando DIFERENTES
            "mediana_RANDOM_3casas": mediana_r,
            "media_XGBOOST_3casas":  media_x,
            "mediana_XGBOOST_3casas": mediana_x,
        })

    df_wilcoxon = pd.DataFrame(resultados)
    return df_wilcoxon


In [ ]:
# dicionário de pares RANDOM / XGBOOST
pares = {
    "LIME":        (df_lime_metrics_RANDOM_calc,        df_lime_metrics_XGBOOST_calc),
    "SHAP":        (df_shap_metrics_RANDOM_calc,        df_shap_metrics_XGBOOST_calc),
    "ANCHOR":      (df_anchor_metrics_RANDOM_calc,      df_anchor_metrics_XGBOOST_calc),
    "SURROGATE":   (df_surrogate_tree_metrics_RANDOM_calc, df_surrogate_tree_metrics_XGBOOST_calc),
    "PERMUTATION": (df_permutation_metrics_RANDOM_calc, df_permutation_metrics_XGBOOST_calc),
}

dfs_wilcoxon = {}        # guarda por método
dfs_wilcoxon_todos = []  # para juntar tudo num só DataFrame, se quiser

for metodo, (df_r, df_x) in pares.items():
    df_w = calcular_wilcoxon_metrics(df_r, df_x, metodo_nome=metodo)
    dfs_wilcoxon[metodo] = df_w
    dfs_wilcoxon_todos.append(df_w)

# DataFrame único com tudo
df_wilcoxon_todos_metodos = pd.concat(dfs_wilcoxon_todos, ignore_index=True)

df_wilcoxon_todos_metodos


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

LIM_DEST = 0.5  # threshold de correlação

def construir_corr_de_wilcoxon(df_w):
    """
    A partir de um df_wilcoxon de UM método, monta:
      - corr_means, labels_means
      - corr_medians, labels_medians
    usando as colunas *_3casas.
    """
    # garantimos que é cópia
    df = df_w.copy()

    # ------- MÉDIAS -------
    col_data_means = {}  # nome_coluna -> vetor [RF, XGB]

    for _, row in df.iterrows():
        met = row["métrica"]
        concl = str(row["conclusão"])

        # tenta converter com segurança
        def to_float_safe(x):
            try:
                return float(str(x).replace(",", "."))
            except Exception:
                return np.nan

        if concl.startswith("Não Diferentes"):
            v = to_float_safe(row["media_geral_3casas"])
            # usa o mesmo valor para RF e XGB
            vec = np.array([v, v], dtype=float)
            col_data_means[met] = vec
        else:
            v_rf  = to_float_safe(row["media_RANDOM_3casas"])
            v_xgb = to_float_safe(row["media_XGBOOST_3casas"])

            col_data_means[f"{met} RF"]  = np.array([v_rf,  np.nan], dtype=float)
            col_data_means[f"{met} XGB"] = np.array([np.nan, v_xgb], dtype=float)

    df_means = pd.DataFrame(col_data_means, index=["RANDOM", "XGBOOST"])
    corr_means = df_means.corr(method="pearson", min_periods=1)
    labels_means = list(corr_means.columns)

    # ------- MEDIANAS -------
    col_data_medians = {}

    for _, row in df.iterrows():
        met = row["métrica"]
        concl = str(row["conclusão"])

        def to_float_safe(x):
            try:
                return float(str(x).replace(",", "."))
            except Exception:
                return np.nan

        if concl.startswith("Não Diferentes"):
            v = to_float_safe(row["mediana_geral_3casas"])
            vec = np.array([v, v], dtype=float)
            col_data_medians[met] = vec
        else:
            v_rf  = to_float_safe(row["mediana_RANDOM_3casas"])
            v_xgb = to_float_safe(row["mediana_XGBOOST_3casas"])

            col_data_medians[f"{met} RF"]  = np.array([v_rf,  np.nan], dtype=float)
            col_data_medians[f"{met} XGB"] = np.array([np.nan, v_xgb], dtype=float)

    df_medians = pd.DataFrame(col_data_medians, index=["RANDOM", "XGBOOST"])
    corr_medians = df_medians.corr(method="pearson", min_periods=1)
    labels_medians = list(corr_medians.columns)

    return corr_means, labels_means, corr_medians, labels_medians


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

LIM_DEST = 0.5  # threshold de correlação

def plot_bw_heatmap_threshold(
    corr_df,
    labels,
    titulo,
    lim_dest=0.5,
    filter_remove=None,  # None, 'xgb' ou 'rf'
    ax=None
):
    """
    'Heatmap' em preto e branco:
      - Mostra matriz completa (triângulo inferior + superior)
      - NÃO mostra valores na diagonal
      - Mostra TODAS as métricas (não remove mais por threshold)
      - filter_remove:
          None -> não remove nada
          'xgb' -> remove métricas cujo nome contenha 'xgb' (case-insensitive)
          'rf'  -> remove métricas cujo nome contenha 'rf'  (case-insensitive)
      - Sem colormap, sem barra de cores
      - Números em preto
      - QUADRADO em volta APENAS quando |r| >= lim_dest
    """
    corr_df = corr_df.copy()
    corr_df.index = labels
    corr_df.columns = labels

    # 0) aplica filtro de remoção (RF/XGB) se solicitado
    if filter_remove is not None:
        labels_trab = [
            name for name in labels
            if filter_remove.lower() not in name.lower()
        ]
    else:
        labels_trab = list(labels)

    if not labels_trab:
        print(f"Nenhuma métrica restante após remover '{filter_remove}' em '{titulo}'.")
        if ax is not None:
            ax.axis("off")
        return

    corr_sub = corr_df.loc[labels_trab, labels_trab]
    labels_sub = labels_trab
    n = len(labels_sub)

    # 2) plotagem
    if ax is None:
        fig_size = max(6, n * 0.6)
        fig, ax = plt.subplots(figsize=(fig_size, fig_size))

    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(n - 0.5, -0.5)  # inverte para parecer matriz

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels_sub, rotation=90, fontsize=7)
    ax.set_yticklabels(labels_sub, fontsize=7)
    ax.set_title(titulo, fontsize=11)

    # grade leve em TODA a matriz
    for i in range(n):
        for j in range(n):
            rect_grid = Rectangle(
                (j - 0.5, i - 0.5),
                1.0, 1.0,
                fill=False,
                edgecolor="0.8",
                linewidth=0.5
            )
            ax.add_patch(rect_grid)

    # escreve TODAS as correlações (fora da diagonal);
    # destaca só as com |r| >= lim_dest
    for i in range(n):
        for j in range(n):
            if i == j:
                continue  # NÃO mostra diagonal

            val = corr_sub.iloc[i, j]
            if np.isnan(val):
                continue

            # escreve o valor
            ax.text(
                j, i,
                f"{val:.2f}",
                ha="center",
                va="center",
                fontsize=9,
                color="black"
            )

            # quadrado em destaque só se passar do threshold
            if abs(val) >= lim_dest:
                rect = Rectangle(
                    (j - 0.5, i - 0.5),
                    1.0, 1.0,
                    fill=False,
                    edgecolor="black",
                    linewidth=1.5
                )
                ax.add_patch(rect)

    ax.tick_params(axis="both", which="both", length=0)


# =============== FIGURA ÚNICA COM 4 PLOTS (2x2) ===============
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Linha 1: MÉDIAS
# Esquerda: RF (remove XGB)
plot_bw_heatmap_threshold(
    corr_means,
    labels_means,
    f"MÉDIAS (|r| ≥ {LIM_DEST}) — RF (sem XGB)",
    lim_dest=LIM_DEST,
    filter_remove="xgb",
    ax=axes[0, 0]
)

# Direita: XGB (remove RF)
plot_bw_heatmap_threshold(
    corr_means,
    labels_means,
    f"MÉDIAS (|r| ≥ {LIM_DEST}) — XGB (sem RF)",
    lim_dest=LIM_DEST,
    filter_remove="rf",
    ax=axes[0, 1]
)

# Linha 2: MEDIANAS
# Esquerda: RF (sem XGB)
plot_bw_heatmap_threshold(
    corr_medians,
    labels_medians,
    f"MEDIANAS (|r| ≥ {LIM_DEST}) — RF (sem XGB)",
    lim_dest=LIM_DEST,
    filter_remove="xgb",
    ax=axes[1, 0]
)

# Direita: XGB (sem RF)
plot_bw_heatmap_threshold(
    corr_medians,
    labels_medians,
    f"MEDIANAS (|r| ≥ {LIM_DEST}) — XGB (sem RF)",
    lim_dest=LIM_DEST,
    filter_remove="rf",
    ax=axes[1, 1]
)

plt.tight_layout()
plt.show()


In [ ]:
for metodo, df_w in dfs_wilcoxon.items():
    print(f"\n================= MÉTODO: {metodo} =================")

    # 1) construir matrizes de correlação (médias e medianas)
    corr_means, labels_means, corr_medians, labels_medians = construir_corr_de_wilcoxon(df_w)

    # 2) figura única com 4 plots (2x2) para ESTE método
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    # Linha 1: MÉDIAS
    # Esquerda: RF (remove XGB)
    low_means_rf = plot_bw_heatmap_threshold(
        corr_means,
        labels_means,
        f"{metodo} — MÉDIAS (|r| ≥ {LIM_DEST}) — RF (sem XGB)",
        lim_dest=LIM_DEST,
        filter_remove="xgb",
        ax=axes[0, 0]
    )

    # Direita: XGB (remove RF)
    low_means_xgb = plot_bw_heatmap_threshold(
        corr_means,
        labels_means,
        f"{metodo} — MÉDIAS (|r| ≥ {LIM_DEST}) — XGB (sem RF)",
        lim_dest=LIM_DEST,
        filter_remove="rf",
        ax=axes[0, 1]
    )

    # Linha 2: MEDIANAS
    # Esquerda: RF (sem XGB)
    low_medians_rf = plot_bw_heatmap_threshold(
        corr_medians,
        labels_medians,
        f"{metodo} — MEDIANAS (|r| ≥ {LIM_DEST}) — RF (sem XGB)",
        lim_dest=LIM_DEST,
        filter_remove="xgb",
        ax=axes[1, 0]
    )

    # Direita: XGB (sem RF)
    low_medians_xgb = plot_bw_heatmap_threshold(
        corr_medians,
        labels_medians,
        f"{metodo} — MEDIANAS (|r| ≥ {LIM_DEST}) — XGB (sem RF)",
        lim_dest=LIM_DEST,
        filter_remove="rf",
        ax=axes[1, 1]
    )

    plt.tight_layout()
    plt.show()

    # (Opcional) montar dataframe com as métricas “fracas” deste método
    rows_low = []
    for m in low_means_rf or []:
        rows_low.append({"metodo": metodo, "grupo": "Médias RF (sem XGB)", "métrica": m})
    for m in low_means_xgb or []:
        rows_low.append({"metodo": metodo, "grupo": "Médias XGB (sem RF)", "métrica": m})
    for m in low_medians_rf or []:
        rows_low.append({"metodo": metodo, "grupo": "Medianas RF (sem XGB)", "métrica": m})
    for m in low_medians_xgb or []:
        rows_low.append({"metodo": metodo, "grupo": "Medianas XGB (sem RF)", "métrica": m})

    df_low_corr_metodo = pd.DataFrame(rows_low, columns=["metodo", "grupo", "métrica"])
    print(df_low_corr_metodo)


In [ ]:
from scipy.stats import wilcoxon
import numpy as np
import pandas as pd

alpha = 0.05  # nível de significância
resultados = []

for col in feat_names:
    # converte para float e garante arrays
    serie_r = df_lime_metrics_RANDOM_calc[col].astype(float).to_numpy()
    serie_x = df_lime_metrics_XGBOOST_calc[col].astype(float).to_numpy()

    # máscara para garantir pares válidos (sem NaN)
    mask_valid = ~np.isnan(serie_r) & ~np.isnan(serie_x)
    serie_r_v = serie_r[mask_valid]
    serie_x_v = serie_x[mask_valid]

    # Wilcoxon pareado
    stat, p = wilcoxon(serie_r_v, serie_x_v, zero_method="wilcox")

    # inicializa campos das novas colunas como vazio
    media_geral   = ""
    mediana_geral = ""
    media_r       = ""
    mediana_r     = ""
    media_x       = ""
    mediana_x     = ""

    # trata NaN explicitamente
    if np.isnan(p):
        p_str = "0"
        conclusao = "Não Diferentes (p ≥ 0.05)"
        p_num = 0.0
    else:
        p_str = f"{p:.4f}"
        p_num = float(f"{p:.4f}")
        if p < alpha:
            conclusao = "Diferentes (p < 0.05)"
        else:
            conclusao = "Não Diferentes (p ≥ 0.05)"

    # === cálculo das médias/medianas com 3 casas ===
    if conclusao.startswith("Não Diferentes"):
        # usa os valores combinados dos dois classificadores
        ambos = np.concatenate([serie_r_v, serie_x_v])
        media_geral   = f"{np.mean(ambos):.3f}"
        mediana_geral = f"{np.median(ambos):.3f}"
    else:
        # usa médias/medianas separadas para cada classificador
        media_r   = f"{np.mean(serie_r_v):.3f}"
        mediana_r = f"{np.median(serie_r_v):.3f}"
        media_x   = f"{np.mean(serie_x_v):.3f}"
        mediana_x = f"{np.median(serie_x_v):.3f}"

    resultados.append({
        "métrica": col,
        "estatística": stat,
        "p-value_num": f"{p_num:.4f}",   # valor numérico (já tratado)
        "conclusão": conclusao,
        # novas colunas
        "media_geral_3casas":   media_geral,    # usada quando NÃO diferentes
        "mediana_geral_3casas": mediana_geral,  # usada quando NÃO diferentes
        "media_RANDOM_3casas":  media_r,        # usadas quando DIFERENTES
        "mediana_RANDOM_3casas": mediana_r,
        "media_XGBOOST_3casas": media_x,
        "mediana_XGBOOST_3casas": mediana_x,
    })

df_wilcoxon_resultados = pd.DataFrame(resultados)

df_wilcoxon_resultados


In [ ]:
from scipy.stats import wilcoxon
import numpy as np
import pandas as pd

alpha = 0.05  # nível de significância
resultados = []

for col in feat_names:
    # converte para float e garante arrays
    serie_r = df_lime_metrics_RANDOM_calc[col].astype(float).to_numpy()
    serie_x = df_lime_metrics_XGBOOST_calc[col].astype(float).to_numpy()

    # máscara para garantir pares válidos (sem NaN)
    mask_valid = ~np.isnan(serie_r) & ~np.isnan(serie_x)
    serie_r_v = serie_r[mask_valid]
    serie_x_v = serie_x[mask_valid]

    # Wilcoxon pareado
    stat, p = wilcoxon(serie_r_v, serie_x_v, zero_method="wilcox")

    # inicializa campos das novas colunas como vazio
    media_geral   = ""
    mediana_geral = ""
    media_r       = ""
    mediana_r     = ""
    media_x       = ""
    mediana_x     = ""

    # trata NaN explicitamente
    if np.isnan(p):
        p_str = "0"
        conclusao = "Não Diferentes (p ≥ 0.05)"
        p_num = 0.0
    else:
        p_str = f"{p:.4f}"
        p_num = float(f"{p:.4f}")
        if p < alpha:
            conclusao = "Diferentes (p < 0.05)"
        else:
            conclusao = "Não Diferentes (p ≥ 0.05)"

    # === cálculo das médias/medianas com 3 casas ===
    if conclusao.startswith("Não Diferentes"):
        # usa os valores combinados dos dois classificadores
        ambos = np.concatenate([serie_r_v, serie_x_v])
        media_geral   = f"{np.mean(ambos):.3f}"
        mediana_geral = f"{np.median(ambos):.3f}"
    else:
        # usa médias/medianas separadas para cada classificador
        media_r   = f"{np.mean(serie_r_v):.3f}"
        mediana_r = f"{np.median(serie_r_v):.3f}"
        media_x   = f"{np.mean(serie_x_v):.3f}"
        mediana_x = f"{np.median(serie_x_v):.3f}"

    resultados.append({
        "métrica": col,
        "estatística": stat,
        "p-value_num": f"{p_num:.4f}",   # valor numérico (já tratado)
        "conclusão": conclusao,
        # novas colunas
        "media_geral_3casas":   media_geral,    # usada quando NÃO diferentes
        "mediana_geral_3casas": mediana_geral,  # usada quando NÃO diferentes
        "media_RANDOM_3casas":  media_r,        # usadas quando DIFERENTES
        "mediana_RANDOM_3casas": mediana_r,
        "media_XGBOOST_3casas": media_x,
        "mediana_XGBOOST_3casas": mediana_x,
    })

df_wilcoxon_resultados = pd.DataFrame(resultados)

df_wilcoxon_resultados


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

LIM_DEST = 0.5  # threshold de correlação

def plot_bw_heatmap_threshold(
    corr_df,
    labels,
    titulo,
    lim_dest=0.5,
    filter_remove=None,  # None, 'xgb' ou 'rf'
    ax=None
):
    """
    'Heatmap' em preto e branco:
      - Mostra matriz completa (triângulo inferior + superior)
      - NÃO mostra valores na diagonal
      - Só mostra células com |r| >= lim_dest
      - Só mantém métricas que tenham pelo menos uma correlação >= lim_dest
      - filter_remove:
          None -> não remove nada
          'xgb' -> remove métricas cujo nome contenha 'xgb' (case-insensitive)
          'rf'  -> remove métricas cujo nome contenha 'rf'  (case-insensitive)
      - Sem colormap, sem barra de cores
      - Números em preto e quadrado em volta da célula selecionada

    Retorna:
      lista de métricas que ficaram FORA do intervalo (ou seja,
      todas as correlações com outras métricas têm |r| < lim_dest).
    """
    corr_df = corr_df.copy()
    corr_df.index = labels
    corr_df.columns = labels

    # 0) aplica filtro de remoção
    if filter_remove is not None:
        labels_trab = [
            name for name in labels
            if filter_remove.lower() not in name.lower()
        ]
    else:
        labels_trab = list(labels)

    if not labels_trab:
        print(f"Nenhuma métrica restante após remover '{filter_remove}' em '{titulo}'.")
        if ax is not None:
            ax.axis("off")
        return []

    corr_df = corr_df.loc[labels_trab, labels_trab]

    # 1) separa:
    #    - keep_labels: têm pelo menos UMA correlação com |r| >= lim_dest
    #    - low_labels: todas as correlações com outras métricas têm |r| < lim_dest
    keep_labels = []
    low_labels = []
    for name in labels_trab:
        s = corr_df.loc[name].drop(name)  # sem diagonal
        if s.abs().ge(lim_dest).any():
            keep_labels.append(name)
        else:
            low_labels.append(name)

    if not keep_labels:
        print(f"Nenhuma correlação com |r| >= {lim_dest} para '{titulo}' após filtro '{filter_remove}'.")
        if ax is not None:
            ax.axis("off")
        # todas as métricas são "fracas"
        return low_labels

    corr_sub = corr_df.loc[keep_labels, keep_labels]
    labels_sub = keep_labels
    n = len(labels_sub)

    # 2) plotagem
    if ax is None:
        fig_size = max(6, n * 0.6)
        fig, ax = plt.subplots(figsize=(fig_size, fig_size))

    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(n - 0.5, -0.5)  # inverte para parecer matriz

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels_sub, rotation=90, fontsize=7)
    ax.set_yticklabels(labels_sub, fontsize=7)
    ax.set_title(titulo, fontsize=11)

    # grade leve em TODA a matriz
    for i in range(n):
        for j in range(n):
            rect_grid = Rectangle(
                (j - 0.5, i - 0.5),
                1.0, 1.0,
                fill=False,
                edgecolor="0.8",
                linewidth=0.5
            )
            ax.add_patch(rect_grid)

    # escreve apenas correlações que passam do threshold (fora da diagonal)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue  # NÃO mostra diagonal

            val = corr_sub.iloc[i, j]
            if np.isnan(val) or abs(val) < lim_dest:
                continue

            ax.text(
                j, i,
                f"{val:.2f}",
                ha="center",
                va="center",
                fontsize=9,
                color="black"
            )

            rect = Rectangle(
                (j - 0.5, i - 0.5),
                1.0, 1.0,
                fill=False,
                edgecolor="black",
                linewidth=1.5
            )
            ax.add_patch(rect)

    ax.tick_params(axis="both", which="both", length=0)

    # retorna as métricas "fracas" (todas correlações com |r| < lim_dest)
    return low_labels


# =============== FIGURA ÚNICA COM 4 PLOTS (2x2) ===============
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Linha 1: MÉDIAS
# Esquerda: RF (remove XGB)
low_means_rf = plot_bw_heatmap_threshold(
    corr_means,
    labels_means,
    f"MÉDIAS (|r| ≥ {LIM_DEST}) — RF (sem XGB)",
    lim_dest=LIM_DEST,
    filter_remove="xgb",
    ax=axes[0, 0]
)

# Direita: XGB (remove RF)
low_means_xgb = plot_bw_heatmap_threshold(
    corr_means,
    labels_means,
    f"MÉDIAS (|r| ≥ {LIM_DEST}) — XGB (sem RF)",
    lim_dest=LIM_DEST,
    filter_remove="rf",
    ax=axes[0, 1]
)

# Linha 2: MEDIANAS
# Esquerda: RF (sem XGB)
low_medians_rf = plot_bw_heatmap_threshold(
    corr_medians,
    labels_medians,
    f"MEDIANAS (|r| ≥ {LIM_DEST}) — RF (sem XGB)",
    lim_dest=LIM_DEST,
    filter_remove="xgb",
    ax=axes[1, 0]
)

# Direita: XGB (sem RF)
low_medians_xgb = plot_bw_heatmap_threshold(
    corr_medians,
    labels_medians,
    f"MEDIANAS (|r| ≥ {LIM_DEST}) — XGB (sem RF)",
    lim_dest=LIM_DEST,
    filter_remove="rf",
    ax=axes[1, 1]
)

plt.tight_layout()
plt.show()

# =============== MOSTRAR MÉTRICAS FORA DO INTERVALO [-0.5, 0.5] ===============
print("\nMÉTRICAS COM TODAS AS CORRELAÇÕES DENTRO DE (-0.5, 0.5):")
print(f"- Médias RF:\n  {low_means_rf}")
print(f"- Médias XGB :\n  {low_means_xgb}")
print(f"- Medianas RF :\n  {low_medians_rf}")
print(f"- Medianas XGB :\n  {low_medians_xgb}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# ------------------- 1) Preparar dados sem NaN (máscara global) -------------------
# garante tipo numérico
df_r_all = df_lime_metrics_RANDOM_calc[feat_names].astype(float).copy()
df_x_all = df_lime_metrics_XGBOOST_calc[feat_names].astype(float).copy()

# máscara global: mantém apenas linhas onde TODAS as métricas têm valor nos dois classificadores
mask_valid_global = (~df_r_all.isna() & ~df_x_all.isna()).all(axis=1)

df_r = df_r_all.loc[mask_valid_global].reset_index(drop=True)
df_x = df_x_all.loc[mask_valid_global].reset_index(drop=True)

# ------------------- 2) Construir variáveis para "médias" e "medianas" -------------------
data_means = {}
data_medians = {}

for col in feat_names:
    serie_r = df_r[col].to_numpy()
    serie_x = df_x[col].to_numpy()

    # pega a conclusão do Wilcoxon para esta métrica
    conclusao = df_wilcoxon_resultados.loc[
        df_wilcoxon_resultados["métrica"] == col, "conclusão"
    ].iloc[0]

    if conclusao.startswith("Não Diferentes"):
        # === Wilcoxon diz que RANDOM e XGBOOST são "iguais" ===
        # Uma variável única com o NOME DA MÉTRICA (ex.: fidelity(%))
        combined_mean = (serie_r + serie_x) / 2.0

        # mediana ponto-a-ponto entre RF e XGB (por instância)
        combined_median = np.median(
            np.vstack([serie_r, serie_x]), axis=0
        )

        data_means[col] = combined_mean
        data_medians[col] = combined_median

    else:
        # === Wilcoxon diz que são "Diferentes" ===
        # Duas variáveis: métrica_RF e métrica_XGB
        data_means[f"{col}_RANDOM"] = serie_r
        data_means[f"{col}_XGBOOST"] = serie_x

        data_medians[f"{col}_RANDOM"] = serie_r
        data_medians[f"{col}_XGBOOST"] = serie_x

# monta DataFrames para correlação (linhas = instâncias, colunas = variáveis do heatmap)
df_means_heat = pd.DataFrame(data_means)
df_medians_heat = pd.DataFrame(data_medians)

# ------------------- 3) Matrizes de correlação de Pearson -------------------
corr_means = df_means_heat.corr(method="pearson")
corr_medians = df_medians_heat.corr(method="pearson")

cols_means = list(df_means_heat.columns)
cols_medians = list(df_medians_heat.columns)

# ------------------- 4) Função para criar labels "bonitos" -------------------
def make_label(col_name: str) -> str:
    """
    Converte:
      'fidelity(%)_RANDOM'  -> 'fidelity(%) \\mathbf{RF}'
      'fidelity(%)_XGBOOST' -> 'fidelity(%) \\mathbf{XGB}'
      'fidelity(%)'         -> 'fidelity(%)'
    usando mathtext para deixar RF/XGB em negrito.
    """
    if col_name.endswith("_RANDOM"):
        base = col_name[:-len("_RANDOM")]
        return rf"{base} $\mathbf{{RF}}$"
    elif col_name.endswith("_XGBOOST"):
        base = col_name[:-len("_XGBOOST")]
        return rf"{base} $\mathbf{{XGB}}$"
    else:
        return col_name

labels_means = [make_label(c) for c in cols_means]
labels_medians = [make_label(c) for c in cols_medians]

# ------------------- 5) Função genérica para plotar um heatmap (apenas triângulo inferior) -------------------
def plot_heatmap_lower_triangle(corr_df, labels, titulo, lim_dest=0.5):
    n = len(labels)

    # aumenta o tamanho da figura em função do número de variáveis
    fig_size = max(6, n * 0.6)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))

    # cria uma cópia e zera (NaN) a parte superior para não exibir
    corr_masked = corr_df.copy()
    for i in range(n):
        for j in range(n):
            if j > i:  # acima da diagonal
                corr_masked.iloc[i, j] = np.nan

    im = ax.imshow(
        corr_masked.values,
        vmin=-1, vmax=1,
        interpolation="nearest",
        aspect="equal",
        cmap="coolwarm"
    )

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_title(titulo, fontsize=11)

    # valores numéricos dentro das células (um pouco maiores agora)
    for i in range(n):
        for j in range(n):
            if j > i:
                continue  # não mostra parte superior
            val = corr_df.iloc[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8)

                # destaque com retângulo se |correlação| >= lim_dest
                if abs(val) >= lim_dest:
                    rect = Rectangle(
                        (j - 0.5, i - 0.5),  # canto inferior esquerdo da célula
                        1.0, 1.0,
                        fill=False,
                        edgecolor="black",
                        linewidth=1.2
                    )
                    ax.add_patch(rect)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Coeficiente de correlação de Pearson", fontsize=9)

    plt.tight_layout()
    plt.show()

# ------------------- 6) Gerar os dois heatmaps em linhas separadas -------------------
plot_heatmap_lower_triangle(
    corr_means,
    labels_means,
    "Correlação de Pearson — combinações de MÉDIAS (RF / XGB)\n(triângulo inferior, |r| ≥ 0.5 destacado)"
)

plot_heatmap_lower_triangle(
    corr_medians,
    labels_medians,
    "Correlação de Pearson — combinações de MEDIANAS (RF / XGB)\n(triângulo inferior, |r| ≥ 0.5 destacado)"
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

LIM_DEST = 0.5  # use o mesmo threshold que você já está usando

def plot_bw_heatmap_threshold(corr_df, labels, titulo, lim_dest=0.5):
    """
    'Heatmap' em preto e branco:
      - Mostra matriz completa (triângulo inferior + superior)
      - NÃO mostra valores na diagonal
      - Só mostra células com |r| >= lim_dest
      - Só mantém métricas que tenham pelo menos uma correlação >= lim_dest
      - Sem colormap, sem barra de cores
      - Números em preto e quadrado em volta da célula selecionada
    """
    # garante rótulos na matriz de correlação
    corr_df = corr_df.copy()
    corr_df.index = labels
    corr_df.columns = labels

    # 1) selecionar apenas métricas que tenham pelo menos UMA correlação |r| >= lim_dest
    keep_labels = []
    for name in labels:
        s = corr_df.loc[name].drop(name)  # remove a diagonal (self)
        if s.abs().ge(lim_dest).any():
            keep_labels.append(name)

    # se nada sobrar, apenas informa e sai
    if not keep_labels:
        print(f"Nenhuma correlação com |r| >= {lim_dest} para '{titulo}'.")
        return

    # submatriz só com métricas relevantes
    corr_sub = corr_df.loc[keep_labels, keep_labels]
    labels_sub = keep_labels
    n = len(labels_sub)

    # 2) plotagem
    fig_size = max(6, n * 0.6)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))

    # limites do "tabuleiro"
    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(n - 0.5, -0.5)  # inverte para parecer matriz

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels_sub, rotation=90, fontsize=7)
    ax.set_yticklabels(labels_sub, fontsize=7)
    ax.set_title(titulo, fontsize=11)

    # grade leve em TODA a matriz
    for i in range(n):
        for j in range(n):
            rect_grid = Rectangle(
                (j - 0.5, i - 0.5),
                1.0, 1.0,
                fill=False,
                edgecolor="0.8",  # cinza claro
                linewidth=0.5
            )
            ax.add_patch(rect_grid)

    # escreve apenas as correlações que passam no threshold (fora da diagonal)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue  # NÃO mostra diagonal

            val = corr_sub.iloc[i, j]
            if np.isnan(val) or abs(val) < lim_dest:
                continue

            # número em preto
            ax.text(
                j, i,
                f"{val:.2f}",
                ha="center",
                va="center",
                fontsize=9,
                color="black"
            )

            # quadrado em destaque em preto
            rect = Rectangle(
                (j - 0.5, i - 0.5),
                1.0, 1.0,
                fill=False,
                edgecolor="black",
                linewidth=1.5
            )
            ax.add_patch(rect)

    ax.tick_params(axis="both", which="both", length=0)
    plt.tight_layout()
    plt.show()

# ---------- usar o plugin para MÉDIAS ----------
plot_bw_heatmap_threshold(
    corr_means,
    labels_means,
    f"Correlação (|r| ≥ {LIM_DEST}) — MÉDIAS (RF/XGB)",
    lim_dest=LIM_DEST
)

# ---------- usar o plugin para MEDIANAS ----------
plot_bw_heatmap_threshold(
    corr_medians,
    labels_medians,
    f"Correlação (|r| ≥ {LIM_DEST}) — MEDIANAS (RF/XGB)",
    lim_dest=LIM_DEST
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

LIM_DEST = 0.5  # use o mesmo threshold que você já está usando

def plot_bw_heatmap_threshold(corr_df, labels, titulo, lim_dest=0.5):
    """
    'Heatmap' em preto e branco:
      - Mostra matriz completa (triângulo inferior + superior)
      - NÃO mostra valores na diagonal
      - Só mostra células com |r| >= lim_dest
      - Só mantém métricas que tenham pelo menos uma correlação >= lim_dest
      - Remove métricas cujo nome contenha 'xgb' (case-insensitive)
      - Sem colormap, sem barra de cores
      - Números em preto e quadrado em volta da célula selecionada
    """
    # ---- 0) Remove métricas que tenham 'xgb' em qualquer lugar do nome ----
    labels_sem_xgb = [
        name for name in labels
        if "xgb" not in name.lower()
    ]

    if not labels_sem_xgb:
        print(f"Nenhuma métrica restante após remover labels com 'XGB' em '{titulo}'.")
        return

    # garante rótulos na matriz de correlação (originais)
    corr_df = corr_df.copy()
    corr_df.index = labels
    corr_df.columns = labels

    # restringe a matriz às métricas sem 'XGB' no nome
    corr_df = corr_df.loc[labels_sem_xgb, labels_sem_xgb]
    labels_trab = labels_sem_xgb

    # 1) selecionar apenas métricas que tenham pelo menos UMA correlação |r| >= lim_dest
    keep_labels = []
    for name in labels_trab:
        s = corr_df.loc[name].drop(name)  # remove a diagonal (self)
        if s.abs().ge(lim_dest).any():
            keep_labels.append(name)

    # se nada sobrar, apenas informa e sai
    if not keep_labels:
        print(f"Nenhuma correlação com |r| >= {lim_dest} para '{titulo}' após filtro 'XGB'.")
        return

    # submatriz só com métricas relevantes
    corr_sub = corr_df.loc[keep_labels, keep_labels]
    labels_sub = keep_labels
    n = len(labels_sub)

    # 2) plotagem
    fig_size = max(6, n * 0.6)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))

    # limites do "tabuleiro"
    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(n - 0.5, -0.5)  # inverte para parecer matriz

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels_sub, rotation=90, fontsize=7)
    ax.set_yticklabels(labels_sub, fontsize=7)
    ax.set_title(titulo, fontsize=11)

    # grade leve em TODA a matriz
    for i in range(n):
        for j in range(n):
            rect_grid = Rectangle(
                (j - 0.5, i - 0.5),
                1.0, 1.0,
                fill=False,
                edgecolor="0.8",  # cinza claro
                linewidth=0.5
            )
            ax.add_patch(rect_grid)

    # escreve apenas as correlações que passam no threshold (fora da diagonal)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue  # NÃO mostra diagonal

            val = corr_sub.iloc[i, j]
            if np.isnan(val) or abs(val) < lim_dest:
                continue

            # número em preto
            ax.text(
                j, i,
                f"{val:.2f}",
                ha="center",
                va="center",
                fontsize=9,
                color="black"
            )

            # quadrado em destaque em preto
            rect = Rectangle(
                (j - 0.5, i - 0.5),
                1.0, 1.0,
                fill=False,
                edgecolor="black",
                linewidth=1.5
            )
            ax.add_patch(rect)

    ax.tick_params(axis="both", which="both", length=0)
    plt.tight_layout()
    plt.show()

# ---------- usar o plugin para MÉDIAS ----------
plot_bw_heatmap_threshold(
    corr_means,
    labels_means,
    f"Correlação (|r| ≥ {LIM_DEST}) — MÉDIAS (RF)",
    lim_dest=LIM_DEST
)

# ---------- usar o plugin para MEDIANAS ----------
plot_bw_heatmap_threshold(
    corr_medians,
    labels_medians,
    f"Correlação (|r| ≥ {LIM_DEST}) — MEDIANAS (RF)",
    lim_dest=LIM_DEST
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

LIM_DEST = 0.5  # use o mesmo threshold que você já está usando

def plot_bw_heatmap_threshold(corr_df, labels, titulo, lim_dest=0.5):
    """
    'Heatmap' em preto e branco:
      - Mostra matriz completa (triângulo inferior + superior)
      - NÃO mostra valores na diagonal
      - Só mostra células com |r| >= lim_dest
      - Só mantém métricas que tenham pelo menos uma correlação >= lim_dest
      - Remove métricas cujo nome contenha 'xgb' (case-insensitive)
      - Sem colormap, sem barra de cores
      - Números em preto e quadrado em volta da célula selecionada
    """
    # ---- 0) Remove métricas que tenham 'xgb' em qualquer lugar do nome ----
    labels_sem_xgb = [
        name for name in labels
        if "rf" not in name.lower()
    ]

    if not labels_sem_xgb:
        print(f"Nenhuma métrica restante após remover labels com 'XGB' em '{titulo}'.")
        return

    # garante rótulos na matriz de correlação (originais)
    corr_df = corr_df.copy()
    corr_df.index = labels
    corr_df.columns = labels

    # restringe a matriz às métricas sem 'XGB' no nome
    corr_df = corr_df.loc[labels_sem_xgb, labels_sem_xgb]
    labels_trab = labels_sem_xgb

    # 1) selecionar apenas métricas que tenham pelo menos UMA correlação |r| >= lim_dest
    keep_labels = []
    for name in labels_trab:
        s = corr_df.loc[name].drop(name)  # remove a diagonal (self)
        if s.abs().ge(lim_dest).any():
            keep_labels.append(name)

    # se nada sobrar, apenas informa e sai
    if not keep_labels:
        print(f"Nenhuma correlação com |r| >= {lim_dest} para '{titulo}' após filtro 'RF'.")
        return

    # submatriz só com métricas relevantes
    corr_sub = corr_df.loc[keep_labels, keep_labels]
    labels_sub = keep_labels
    n = len(labels_sub)

    # 2) plotagem
    fig_size = max(6, n * 0.6)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))

    # limites do "tabuleiro"
    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(n - 0.5, -0.5)  # inverte para parecer matriz

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels_sub, rotation=90, fontsize=7)
    ax.set_yticklabels(labels_sub, fontsize=7)
    ax.set_title(titulo, fontsize=11)

    # grade leve em TODA a matriz
    for i in range(n):
        for j in range(n):
            rect_grid = Rectangle(
                (j - 0.5, i - 0.5),
                1.0, 1.0,
                fill=False,
                edgecolor="0.8",  # cinza claro
                linewidth=0.5
            )
            ax.add_patch(rect_grid)

    # escreve apenas as correlações que passam no threshold (fora da diagonal)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue  # NÃO mostra diagonal

            val = corr_sub.iloc[i, j]
            if np.isnan(val) or abs(val) < lim_dest:
                continue

            # número em preto
            ax.text(
                j, i,
                f"{val:.2f}",
                ha="center",
                va="center",
                fontsize=9,
                color="black"
            )

            # quadrado em destaque em preto
            rect = Rectangle(
                (j - 0.5, i - 0.5),
                1.0, 1.0,
                fill=False,
                edgecolor="black",
                linewidth=1.5
            )
            ax.add_patch(rect)

    ax.tick_params(axis="both", which="both", length=0)
    plt.tight_layout()
    plt.show()

# ---------- usar o plugin para MÉDIAS ----------
plot_bw_heatmap_threshold(
    corr_means,
    labels_means,
    f"Correlação (|r| ≥ {LIM_DEST}) — MÉDIAS (XGB)",
    lim_dest=LIM_DEST
)

# ---------- usar o plugin para MEDIANAS ----------
plot_bw_heatmap_threshold(
    corr_medians,
    labels_medians,
    f"Correlação (|r| ≥ {LIM_DEST}) — MEDIANAS (XGB)",
    lim_dest=LIM_DEST
)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.stats import wilcoxon

# ============================================================
# 1) Configurações e listas de métricas
# ============================================================

feat_names = [
    "fidelity(%)","infidelity(%)","faithfulness(%)","simplicity(%)",
    "consistency(%)","robustez(%)","completeness(%)","selectivity(%)",
    "soundness(%)","directional soundness(%)","stability(%)",
    "sufficiency-1(%)","sufficiency-5(%)","sufficiency-10(%)","sufficiency-20(%)"
]

ALPHA = 0.05       # nível de significância do Wilcoxon
LIM_DEST = 0.5     # threshold de correlação para destacar

# ============================================================
# 2) Helper para preparar *_calc (remover últimas 2 linhas)
#    (ajuste o nome dos dataframes originais se precisar)
# ============================================================

def preparar_metrics_calc(df):
    df_calc = df[feat_names].copy()
    # Remove as duas últimas linhas (MÉDIA / MEDIANA ou agregados)
    df_calc = df_calc.iloc[:-2].copy()
    return df_calc

# Exemplo: assumindo que estes dataframes já existem no seu ambiente
# df_lime_metrics_RANDOM, df_lime_metrics_XGBOOST, etc.

df_lime_metrics_RANDOM_calc        = preparar_metrics_calc(df_lime_metrics_RANDOM)
df_lime_metrics_XGBOOST_calc       = preparar_metrics_calc(df_lime_metrics_XGBOOST)

df_shape_metrics_RANDOM_calc        = preparar_metrics_calc(df_shape_metrics_RANDOM)
df_shape_metrics_XGBOOST_calc       = preparar_metrics_calc(df_shape_metrics_XGBOOST)

df_anchor_metrics_RANDOM_calc      = preparar_metrics_calc(df_anchor_metrics_RANDOM)
df_anchor_metrics_XGBOOST_calc     = preparar_metrics_calc(df_anchor_metrics_XGBOOST)

df_surrogate_tree_metrics_RANDOM_calc  = preparar_metrics_calc(df_surrogate_tree_metrics_RANDOM)
df_surrogate_tree_metrics_XGBOOST_calc = preparar_metrics_calc(df_surrogate_tree_metrics_XGBOOST)

df_permutation_metrics_RANDOM_calc     = preparar_metrics_calc(df_permutation_metrics_RANDOM)
df_permutation_metrics_XGBOOST_calc    = preparar_metrics_calc(df_permutation_metrics_XGBOOST)

# Dicionário de pares RANDOM / XGBOOST para facilitar o loop
pares_raw = {
    "LIME":        (df_lime_metrics_RANDOM_calc,        df_lime_metrics_XGBOOST_calc),
    "SHAP":        (df_shap_metrics_RANDOM_calc,        df_shap_metrics_XGBOOST_calc),
    "ANCHOR":      (df_anchor_metrics_RANDOM_calc,      df_anchor_metrics_XGBOOST_calc),
    "SURROGATE":   (df_surrogate_tree_metrics_RANDOM_calc, df_surrogate_tree_metrics_XGBOOST_calc),
    "PERMUTATION": (df_permutation_metrics_RANDOM_calc, df_permutation_metrics_XGBOOST_calc),
}

# ============================================================
# 3) Função: Wilcoxon + médias/medianas por método
#    (resultado por métrica: iguais x diferentes)
# ============================================================

def calcular_wilcoxon_metrics(df_random_calc, df_xgboost_calc, metodo_nome):
    resultados = []

    for col in feat_names:
        # converte para float e garante arrays
        serie_r = df_random_calc[col].astype(float).to_numpy()
        serie_x = df_xgboost_calc[col].astype(float).to_numpy()

        # máscara para garantir pares válidos (sem NaN)
        mask_valid = ~np.isnan(serie_r) & ~np.isnan(serie_x)
        serie_r_v = serie_r[mask_valid]
        serie_x_v = serie_x[mask_valid]

        if len(serie_r_v) == 0:
            continue  # sem dados válidos

        # Wilcoxon pareado
        stat, p = wilcoxon(serie_r_v, serie_x_v, zero_method="wilcox")

        # inicializa campos das novas colunas como vazio
        media_geral   = ""
        mediana_geral = ""
        media_r       = ""
        mediana_r     = ""
        media_x       = ""
        mediana_x     = ""

        # trata NaN explicitamente
        if np.isnan(p):
            p_num = 0.0
            conclusao = "Não Diferentes (p ≥ 0.05)"
        else:
            p_num = float(f"{p:.4f}")
            if p < ALPHA:
                conclusao = "Diferentes (p < 0.05)"
            else:
                conclusao = "Não Diferentes (p ≥ 0.05)"

        # cálculo das médias/medianas com 3 casas (para registro, se quiser)
        if conclusao.startswith("Não Diferentes"):
            ambos = np.concatenate([serie_r_v, serie_x_v])
            media_geral   = f"{np.mean(ambos):.3f}"
            mediana_geral = f"{np.median(ambos):.3f}"
        else:
            media_r   = f"{np.mean(serie_r_v):.3f}"
            mediana_r = f"{np.median(serie_r_v):.3f}"
            media_x   = f"{np.mean(serie_x_v):.3f}"
            mediana_x = f"{np.median(serie_x_v):.3f}"

        resultados.append({
            "metodo": metodo_nome,
            "métrica": col,
            "estatística": stat,
            "p-value_num": f"{p_num:.4f}",
            "conclusao": conclusao,
            "media_geral_3casas":    media_geral,
            "mediana_geral_3casas":  mediana_geral,
            "media_RANDOM_3casas":   media_r,
            "mediana_RANDOM_3casas": mediana_r,
            "media_XGBOOST_3casas":  media_x,
            "mediana_XGBOOST_3casas": mediana_x,
        })

    df_w = pd.DataFrame(resultados)
    return df_w

# Calcula df_wilcoxon para todos os métodos
dfs_wilcoxon = {}
for metodo, (df_r, df_x) in pares_raw.items():
    dfs_wilcoxon[metodo] = calcular_wilcoxon_metrics(df_r, df_x, metodo_nome=metodo)

# ============================================================
# 4) Construir matriz de correlação a partir dos dados por instância
#    usando a lógica do Wilcoxon (igual/diferente)
# ============================================================

def construir_corr_from_raw(df_random_calc, df_xgboost_calc, df_w):
    """
    Usa as métricas por instância para montar um DataFrame de features:
      - Se "Não Diferentes": coluna única 'métrica' = média por instância de (RF, XGB)
      - Se "Diferentes": duas colunas 'métrica RF', 'métrica XGB'
    Em seguida, retorna corr_df (Pearson) e labels.
    """
    col_dict = {}

    for _, row in df_w.iterrows():
        met = row["métrica"]
        concl = str(row["conclusao"])

        # séries por instância (sem cortar linhas aqui!)
        s_r = df_random_calc[met].astype(float)
        s_x = df_xgboost_calc[met].astype(float)

        if concl.startswith("Não Diferentes"):
            # coluna única com média dos dois classificadores por instância
            s_mean = (s_r + s_x) / 2.0
            col_dict[met] = s_mean
        else:
            col_dict[f"{met} RF"]  = s_r
            col_dict[f"{met} XGB"] = s_x

    df_feat = pd.DataFrame(col_dict)

    # matriz de correlação de Pearson entre as colunas
    corr_df = df_feat.corr(method="pearson", min_periods=5)
    labels = list(corr_df.columns)
    return corr_df, labels

# ============================================================
# 5) Função de heatmap P&B com destaque no LIM_DEST
# ============================================================

def plot_bw_heatmap_threshold(
    corr_df,
    labels,
    titulo,
    lim_dest=0.5,
    filter_remove=None,  # None, 'xgb' ou 'rf'
    ax=None
):
    """
    'Heatmap' em preto e branco textual:
      - Mostra matriz completa (triângulo inferior + superior)
      - NÃO mostra valores na diagonal
      - Mostra todos os valores (inclusive com |r| < lim_dest)
      - filter_remove:
          None -> não remove nada
          'xgb' -> remove métricas cujo nome contenha 'xgb' (case-insensitive)
          'rf'  -> remove métricas cujo nome contenha 'rf'  (case-insensitive)
      - Sem colormap
      - Números em preto
      - QUADRADO em volta APENAS quando |r| >= lim_dest
    """
    corr_df = corr_df.copy()
    corr_df.index = labels
    corr_df.columns = labels

    # 0) aplica filtro RF/XGB se solicitado
    if filter_remove is not None:
        labels_trab = [
            name for name in labels
            if filter_remove.lower() not in name.lower()
        ]
    else:
        labels_trab = list(labels)

    if not labels_trab:
        print(f"Nenhuma métrica restante após remover '{filter_remove}' em '{titulo}'.")
        if ax is not None:
            ax.axis("off")
        return

    corr_sub = corr_df.loc[labels_trab, labels_trab]
    labels_sub = labels_trab
    n = len(labels_sub)

    if ax is None:
        fig_size = max(6, n * 0.6)
        fig, ax = plt.subplots(figsize=(fig_size, fig_size))

    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(n - 0.5, -0.5)

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels_sub, rotation=90, fontsize=7)
    ax.set_yticklabels(labels_sub, fontsize=7)
    ax.set_title(titulo, fontsize=11)

    # grade leve
    for i in range(n):
        for j in range(n):
            rect_grid = Rectangle(
                (j - 0.5, i - 0.5),
                1.0, 1.0,
                fill=False,
                edgecolor="0.8",
                linewidth=0.5
            )
            ax.add_patch(rect_grid)

    # escreve valores e destaca com quadrado se |r| >= lim_dest
    for i in range(n):
        for j in range(n):
            if i == j:
                continue  # sem diagonal

            val = corr_sub.iloc[i, j]
            if np.isnan(val):
                continue

            ax.text(
                j, i,
                f"{val:.2f}",
                ha="center",
                va="center",
                fontsize=9,
                color="black"
            )

            if abs(val) >= lim_dest:
                rect = Rectangle(
                    (j - 0.5, i - 0.5),
                    1.0, 1.0,
                    fill=False,
                    edgecolor="black",
                    linewidth=1.5
                )
                ax.add_patch(rect)

    ax.tick_params(axis="both", which="both", length=0)

# ============================================================
# 6) Loop final: gerar HEATMAPS para cada método
#    (1 linha, 2 colunas: sem XGB / sem RF)
# ============================================================

for metodo, (df_r, df_x) in pares_raw.items():
    print(f"\n================ HEATMAPS — MÉTODO: {metodo} ================")

    df_w = dfs_wilcoxon[metodo]

    # constrói matriz de correlação usando dados por instância + lógica do Wilcoxon
    corr_all, labels_all = construir_corr_from_raw(df_r, df_x, df_w)

    # figura com 2 heatmaps lado a lado
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Esquerda: sem XGB (foco em RF + métricas "iguais")
    plot_bw_heatmap_threshold(
        corr_all,
        labels_all,
        f"{metodo} — Correlação (|r| ≥ {LIM_DEST}) — SEM XGB",
        lim_dest=LIM_DEST,
        filter_remove="xgb",
        ax=axes[0]
    )

    # Direita: sem RF (foco em XGB + métricas "iguais")
    plot_bw_heatmap_threshold(
        corr_all,
        labels_all,
        f"{metodo} — Correlação (|r| ≥ {LIM_DEST}) — SEM RF",
        lim_dest=LIM_DEST,
        filter_remove="rf",
        ax=axes[1]
    )

    plt.tight_layout()
    plt.show()
